In [ ]:
# ================================================================
# PHASE 3C — CELL 1
# MOUNT GOOGLE DRIVE
# ================================================================

from google.colab import drive

drive.mount(
    "/content/drive"
)

print("✓ Google Drive mounted.")

Mounted at /content/drive
✓ Google Drive mounted.


In [ ]:
# ================================================================
# PHASE 3C — CELL 2
# IMPORTS AND REPRODUCIBILITY
# ================================================================

import os
import json
import random
import hashlib
import warnings

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import cv2
import nibabel as nib

from scipy import ndimage
from scipy.ndimage import binary_fill_holes

import tensorflow as tf

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    cohen_kappa_score
)

warnings.filterwarnings("ignore")


# ---------------------------------------------------------------
# REPRODUCIBILITY
# ---------------------------------------------------------------

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


print("=" * 72)
print("PHASE 3C — LEARNED CLASSIFIER DEVELOPMENT")
print("=" * 72)

print("\nRandom seed :", SEED)
print("TensorFlow :", tf.__version__)

print(
    "\n✓ Imports loaded and reproducibility settings initialized."
)

PHASE 3C — LEARNED CLASSIFIER DEVELOPMENT

Random seed : 42
TensorFlow : 2.20.0

✓ Imports loaded and reproducibility settings initialized.


In [ ]:
# ================================================================
# PHASE 3C — CELL 3
# PROJECT PATHS AND FRESH PHASE 3C DIRECTORY
# ================================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/CAMUS_FINAL"
)

PHASE1_DIR = (
    PROJECT_DIR /
    "phase1_segmentation"
)

PHASE2_DIR = (
    PROJECT_DIR /
    "phase2_quantitative"
)

PHASE3_DIR = (
    PROJECT_DIR /
    "phase3_classification"
)


# ---------------------------------------------------------------
# EXISTING FROZEN PHASES
# ---------------------------------------------------------------

PHASE3A_DIR = (
    PHASE3_DIR /
    "phase3A_label_audit"
)

PHASE3B_DIR = (
    PHASE3_DIR /
    "phase3B_direct_baseline"
)

PHASE3D_DIR = (
    PHASE3_DIR /
    "phase3D_final_evaluation"
)


# ---------------------------------------------------------------
# NEW PHASE 3C DIRECTORY
# ---------------------------------------------------------------

PHASE3C_DIR = (
    PHASE3_DIR /
    "phase3C_classifier_development"
)

PHASE3C_CONFIG_DIR = (
    PHASE3C_DIR /
    "configs"
)

PHASE3C_TABLE_DIR = (
    PHASE3C_DIR /
    "tables"
)

PHASE3C_MODEL_DIR = (
    PHASE3C_DIR /
    "models"
)

PHASE3C_PLOT_DIR = (
    PHASE3C_DIR /
    "plots"
)


for directory in [
    PHASE3C_DIR,
    PHASE3C_CONFIG_DIR,
    PHASE3C_TABLE_DIR,
    PHASE3C_MODEL_DIR,
    PHASE3C_PLOT_DIR
]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# ---------------------------------------------------------------
# BASIC PROJECT CHECK
# ---------------------------------------------------------------

required_existing_dirs = {

    "Phase 1":
        PHASE1_DIR,

    "Phase 2":
        PHASE2_DIR,

    "Phase 3A":
        PHASE3A_DIR,

    "Phase 3B":
        PHASE3B_DIR
}


print("=" * 72)
print("PROJECT DIRECTORY CHECK")
print("=" * 72)


for name, path in required_existing_dirs.items():

    status = (
        "FOUND"
        if path.exists()
        else "MISSING"
    )

    print(
        f"{name:<10}: {status:<7} | {path}"
    )


print("\n" + "=" * 72)
print("NEW PHASE 3C STRUCTURE")
print("=" * 72)

print(
    PHASE3C_DIR
)

for directory in [
    PHASE3C_CONFIG_DIR,
    PHASE3C_TABLE_DIR,
    PHASE3C_MODEL_DIR,
    PHASE3C_PLOT_DIR
]:

    print(
        "  ✓",
        directory.name
    )


# ---------------------------------------------------------------
# HARD SAFETY CHECK
# ---------------------------------------------------------------

assert PROJECT_DIR.exists(), (
    f"Project directory missing: {PROJECT_DIR}"
)

assert PHASE1_DIR.exists(), (
    "Phase 1 directory is missing."
)

assert PHASE2_DIR.exists(), (
    "Phase 2 directory is missing."
)

assert PHASE3A_DIR.exists(), (
    "Phase 3A directory is missing."
)

assert PHASE3B_DIR.exists(), (
    "Phase 3B directory is missing."
)


print(
    "\n✓ Fresh Phase 3C workspace created successfully."
)

print(
    "✓ Existing Phase 1, Phase 2, Phase 3A and Phase 3B "
    "directories were not modified."
)

PROJECT DIRECTORY CHECK
Phase 1   : FOUND   | /content/drive/MyDrive/CAMUS_FINAL/phase1_segmentation
Phase 2   : FOUND   | /content/drive/MyDrive/CAMUS_FINAL/phase2_quantitative
Phase 3A  : FOUND   | /content/drive/MyDrive/CAMUS_FINAL/phase3_classification/phase3A_label_audit
Phase 3B  : FOUND   | /content/drive/MyDrive/CAMUS_FINAL/phase3_classification/phase3B_direct_baseline

NEW PHASE 3C STRUCTURE
/content/drive/MyDrive/CAMUS_FINAL/phase3_classification/phase3C_classifier_development
  ✓ configs
  ✓ tables
  ✓ models
  ✓ plots

✓ Fresh Phase 3C workspace created successfully.
✓ Existing Phase 1, Phase 2, Phase 3A and Phase 3B directories were not modified.


In [ ]:
# ================================================================
# PHASE 3C — CELL 4
# VERIFY REQUIRED FROZEN ARTIFACTS
# ================================================================

# ---------------------------------------------------------------
# PHASE 1 — FINAL SEGMENTATION MODEL
# ---------------------------------------------------------------

FINAL_SEG_MODEL_PATH = (
    PHASE1_DIR
    / "phase1F_final_model"
    / "models"
    / "final_lv_segmentation_model.keras"
)


# ---------------------------------------------------------------
# PHASE 2 — FROZEN RECORDS / TEST RESULTS
# ---------------------------------------------------------------

PHASE2_FINAL_RECORD_PATH = (
    PHASE2_DIR
    / "PHASE2_FINAL_RECORD.json"
)

PHASE2_COMPLETE_PATH = (
    PHASE2_DIR
    / "PHASE2_COMPLETE.json"
)

PHASE2_METHOD_DECISION_PATH = (
    PHASE2_DIR
    / "phase2A_measurement_method"
    / "configs"
    / "phase2A_geometry_method_decision.json"
)

PHASE2_TEST_TABLE_PATH = (
    PHASE2_DIR
    / "phase2B_final_evaluation"
    / "tables"
    / "final_test_quantitative_measurements.csv"
)


# ---------------------------------------------------------------
# PHASE 3A — FROZEN REFERENCE LABELS
# ---------------------------------------------------------------

PHASE3A_LABELS_PATH = (
    PHASE3A_DIR
    / "tables"
    / "phase3_reference_EF_labels.csv"
)

PHASE3A_DECISION_PATH = (
    PHASE3A_DIR
    / "configs"
    / "phase3A_classification_target_decision.json"
)


# ---------------------------------------------------------------
# PHASE 3B — FROZEN DIRECT BASELINE
# ---------------------------------------------------------------

PHASE3B_RECORD_PATH = (
    PHASE3B_DIR
    / "configs"
    / "phase3B_direct_baseline_record.json"
)


# ---------------------------------------------------------------
# CHECK EVERYTHING
# ---------------------------------------------------------------

required_artifacts = {
    "Final segmentation model":
        FINAL_SEG_MODEL_PATH,

    "Phase 2 final record":
        PHASE2_FINAL_RECORD_PATH,

    "Phase 2 completion marker":
        PHASE2_COMPLETE_PATH,

    "Phase 2 geometry decision":
        PHASE2_METHOD_DECISION_PATH,

    "Phase 2 final test table":
        PHASE2_TEST_TABLE_PATH,

    "Phase 3A reference labels":
        PHASE3A_LABELS_PATH,

    "Phase 3A target decision":
        PHASE3A_DECISION_PATH,

    "Phase 3B direct baseline record":
        PHASE3B_RECORD_PATH
}


print("=" * 80)
print("FROZEN ARTIFACT VERIFICATION")
print("=" * 80)


all_found = True

for name, path in required_artifacts.items():

    exists = path.exists()

    status = (
        "✓ FOUND"
        if exists
        else "✗ MISSING"
    )

    print(
        f"{status:<10} | {name}"
    )

    if not exists:
        print(
            f"             {path}"
        )

        all_found = False


print("\n" + "=" * 80)


if all_found:

    print(
        "✓ All required frozen artifacts are available."
    )

    print(
        "✓ Safe to continue Phase 3C development."
    )

else:

    print(
        "✗ One or more required artifacts are missing."
    )

    print(
        "STOP — do not continue to the next phase."
    )

FROZEN ARTIFACT VERIFICATION
✓ FOUND    | Final segmentation model
✓ FOUND    | Phase 2 final record
✓ FOUND    | Phase 2 completion marker
✓ FOUND    | Phase 2 geometry decision
✓ FOUND    | Phase 2 final test table
✓ FOUND    | Phase 3A reference labels
✓ FOUND    | Phase 3A target decision
✓ FOUND    | Phase 3B direct baseline record

✓ All required frozen artifacts are available.
✓ Safe to continue Phase 3C development.


In [ ]:
# ================================================================
# PHASE 3C — CELL 5
# LOAD AND VERIFY FROZEN PHASE 3A LABELS
# ================================================================

phase3_labels_df = pd.read_csv(
    PHASE3A_LABELS_PATH
)


print("=" * 80)
print("PHASE 3A — FROZEN EF CLASSIFICATION LABELS")
print("=" * 80)


print(
    "\nTable shape:",
    phase3_labels_df.shape
)


print(
    "\nColumns:"
)

for column in phase3_labels_df.columns:
    print(" •", column)


# ---------------------------------------------------------------
# BASIC CHECKS
# ---------------------------------------------------------------

required_columns = {
    "patient_id",
    "split",
    "reference_EF_pct",
    "reference_EF_class"
}


missing_columns = (
    required_columns
    -
    set(
        phase3_labels_df.columns
    )
)


assert len(missing_columns) == 0, (
    f"Missing required columns: {missing_columns}"
)


assert (
    phase3_labels_df["patient_id"]
    .duplicated()
    .sum()
    ==
    0
), "Duplicate patient IDs detected."


assert (
    phase3_labels_df[
        "reference_EF_pct"
    ]
    .notna()
    .all()
), "Missing reference EF values detected."


assert (
    phase3_labels_df[
        "reference_EF_class"
    ]
    .notna()
    .all()
), "Missing reference EF classes detected."


# ---------------------------------------------------------------
# SPLIT COUNTS
# ---------------------------------------------------------------

print(
    "\nPatient counts by split:"
)

split_counts = (
    phase3_labels_df[
        "split"
    ]
    .value_counts()
)

print(
    split_counts.to_string()
)


# ---------------------------------------------------------------
# CLASS DISTRIBUTION BY SPLIT
# ---------------------------------------------------------------

class_distribution = pd.crosstab(
    phase3_labels_df[
        "split"
    ],
    phase3_labels_df[
        "reference_EF_class"
    ]
)


print(
    "\nEF class distribution:"
)

display(
    class_distribution
)


# ---------------------------------------------------------------
# EXPECTED SPLIT CHECK
# ---------------------------------------------------------------

expected_counts = {
    "train": 400,
    "validation": 50,
    "test": 50
}


for split_name, expected_n in expected_counts.items():

    actual_n = int(
        (
            phase3_labels_df[
                "split"
            ]
            ==
            split_name
        ).sum()
    )

    assert actual_n == expected_n, (
        f"{split_name}: expected "
        f"{expected_n}, found {actual_n}"
    )


# ---------------------------------------------------------------
# PHASE 3C DEVELOPMENT DATA ONLY
# ---------------------------------------------------------------

phase3C_labels_df = (
    phase3_labels_df[
        phase3_labels_df[
            "split"
        ].isin(
            [
                "train",
                "validation"
            ]
        )
    ]
    .copy()
    .reset_index(drop=True)
)


assert len(
    phase3C_labels_df
) == 450


assert not (
    phase3C_labels_df[
        "split"
    ]
    ==
    "test"
).any()


print("\n" + "=" * 80)

print(
    "✓ Frozen Phase 3A labels verified."
)

print(
    "✓ Train patients      :",
    int(
        (
            phase3C_labels_df["split"]
            ==
            "train"
        ).sum()
    )
)

print(
    "✓ Validation patients :",
    int(
        (
            phase3C_labels_df["split"]
            ==
            "validation"
        ).sum()
    )
)

print(
    "✓ Test patients used  : 0"
)

print(
    "\n✓ Phase 3C development dataframe contains "
    "only train + validation patients."
)

PHASE 3A — FROZEN EF CLASSIFICATION LABELS

Table shape: (500, 7)

Columns:
 • patient_id
 • split
 • reference_EF_pct
 • reference_EF_class
 • age
 • sex
 • image_quality

Patient counts by split:
split
train         400
test           50
validation     50

EF class distribution:


reference_EF_class,Higher_EF,Intermediate_EF,Lower_EF
split,,,
test,20,14,16
train,132,125,143
validation,20,17,13



✓ Frozen Phase 3A labels verified.
✓ Train patients      : 400
✓ Validation patients : 50
✓ Test patients used  : 0

✓ Phase 3C development dataframe contains only train + validation patients.


In [ ]:
# ================================================================
# PHASE 3C — CELL 6
# PREDEFINED DEVELOPMENT PROTOCOL
# ================================================================

PHASE3C_PROTOCOL_PATH = (
    PHASE3C_CONFIG_DIR /
    "phase3C_predefined_protocol.json"
)


phase3C_protocol = {

    "phase":
        "Phase 3C — Learned Classifier Development",

    "purpose":
        (
            "Develop a lightweight classifier using frozen "
            "quantitative LV measurements to predict the frozen "
            "Phase 3A EF-based functional categories."
        ),

    "classification_target":
        "Phase 3A frozen 3-class reference-EF target",

    "classes": [
        "Lower_EF",
        "Intermediate_EF",
        "Higher_EF"
    ],

    "class_definition": {
        "Lower_EF":
            "reference EF <= 40%",

        "Intermediate_EF":
            "40% < reference EF < 50%",

        "Higher_EF":
            "reference EF >= 50%"
    },

    "development_partitions": {
        "training":
            400,

        "validation":
            50,

        "test_used_during_phase3C":
            0
    },

    "primary_selection_metric":
        "validation macro F1",

    "secondary_metrics": [
        "accuracy",
        "balanced accuracy",
        "weighted F1",
        "Cohen kappa",
        "per-class precision",
        "per-class recall",
        "per-class F1"
    ],

    "feature_sets": {

        "F1_EF_only": [
            "PRED_EF_pct"
        ],

        "F2_volumetric": [
            "PRED_EDV_ml",
            "PRED_ESV_ml",
            "PRED_SV_ml",
            "PRED_EF_pct"
        ],

        "F3_full_quantitative": [
            "PRED_ED_area_mm2",
            "PRED_ES_area_mm2",
            "PRED_ED_length_mm",
            "PRED_ES_length_mm",
            "PRED_EDV_ml",
            "PRED_ESV_ml",
            "PRED_SV_ml",
            "PRED_EF_pct"
        ]
    },

    "classifier":
        "StandardScaler + multinomial-capable LogisticRegression",

    "classifier_settings": {
        "solver":
            "lbfgs",

        "max_iter":
            5000,

        "random_state":
            42
    },

    "resampling":
        "none",

    "SMOTE":
        False,

    "class_weights_predefined":
        None,

    "feature_scaling":
        (
            "StandardScaler fitted on training data only "
            "within sklearn Pipeline."
        ),

    "segmentation_model_frozen":
        True,

    "quantitative_pipeline_frozen":
        True,

    "phase2_geometry_method":
        "PCA major-axis + single-plane area-length",

    "test_data_allowed_for_model_selection":
        False,

    "selection_rule":
        (
            "Select the feature set/classifier configuration "
            "with the highest validation macro F1. "
            "The final test partition is reserved for Phase 3D."
        ),

    "clinical_framing":
        (
            "EF-based functional classification for an "
            "assistive, non-diagnostic application."
        )
}


with open(
    PHASE3C_PROTOCOL_PATH,
    "w"
) as file:

    json.dump(
        phase3C_protocol,
        file,
        indent=2
    )


print("=" * 80)
print("PHASE 3C PREDEFINED PROTOCOL")
print("=" * 80)

print(
    "\nPrimary selection metric :",
    phase3C_protocol[
        "primary_selection_metric"
    ]
)

print(
    "Training patients        : 400"
)

print(
    "Validation patients      : 50"
)

print(
    "Test patients            : 0 during Phase 3C"
)

print(
    "\nFeature sets:"
)

for name, features in (
    phase3C_protocol[
        "feature_sets"
    ].items()
):

    print(
        f"\n{name}:"
    )

    for feature in features:
        print("  •", feature)


print(
    "\n✓ Protocol saved:"
)

print(
    PHASE3C_PROTOCOL_PATH
)

PHASE 3C PREDEFINED PROTOCOL

Primary selection metric : validation macro F1
Training patients        : 400
Validation patients      : 50
Test patients            : 0 during Phase 3C

Feature sets:

F1_EF_only:
  • PRED_EF_pct

F2_volumetric:
  • PRED_EDV_ml
  • PRED_ESV_ml
  • PRED_SV_ml
  • PRED_EF_pct

F3_full_quantitative:
  • PRED_ED_area_mm2
  • PRED_ES_area_mm2
  • PRED_ED_length_mm
  • PRED_ES_length_mm
  • PRED_EDV_ml
  • PRED_ESV_ml
  • PRED_SV_ml
  • PRED_EF_pct

✓ Protocol saved:
/content/drive/MyDrive/CAMUS_FINAL/phase3_classification/phase3C_classifier_development/configs/phase3C_predefined_protocol.json


In [ ]:
# ================================================================
# PHASE 3C — CELL 7
# LOCATE / EXTRACT CAMUS DATASET
# ================================================================

import zipfile


OUTER_ZIP_PATH = Path(
    "/content/drive/MyDrive/download.zip"
)

OUTER_EXTRACT_DIR = Path(
    "/content/camus_data"
)

INNER_EXTRACT_DIR = Path(
    "/content/camus_data_final"
)


# ---------------------------------------------------------------
# OUTER ARCHIVE
# ---------------------------------------------------------------

if not OUTER_EXTRACT_DIR.exists():

    print(
        "Extracting outer CAMUS archive..."
    )

    OUTER_EXTRACT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    with zipfile.ZipFile(
        OUTER_ZIP_PATH,
        "r"
    ) as archive:

        archive.extractall(
            OUTER_EXTRACT_DIR
        )


# ---------------------------------------------------------------
# INNER ARCHIVE
# ---------------------------------------------------------------

INNER_ARCHIVE_PATH = (
    OUTER_EXTRACT_DIR /
    "download"
)


if not INNER_EXTRACT_DIR.exists():

    print(
        "Extracting inner CAMUS archive..."
    )

    INNER_EXTRACT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    with zipfile.ZipFile(
        INNER_ARCHIVE_PATH,
        "r"
    ) as archive:

        archive.extractall(
            INNER_EXTRACT_DIR
        )


# ---------------------------------------------------------------
# FINAL DATASET PATH
# ---------------------------------------------------------------

DATASET_PATH = (
    INNER_EXTRACT_DIR /
    "database_nifti"
)


assert DATASET_PATH.exists(), (
    f"CAMUS dataset not found: {DATASET_PATH}"
)


patient_dirs = sorted([
    path
    for path in DATASET_PATH.iterdir()
    if (
        path.is_dir()
        and path.name.startswith(
            "patient"
        )
    )
])


print("=" * 80)
print("CAMUS DATASET CHECK")
print("=" * 80)

print(
    "\nDataset path:"
)

print(
    DATASET_PATH
)

print(
    "\nPatient directories found:",
    len(patient_dirs)
)


assert len(patient_dirs) == 500, (
    f"Expected 500 patients, found "
    f"{len(patient_dirs)}"
)


print(
    "\n✓ All 500 CAMUS patient directories found."
)

Extracting outer CAMUS archive...
Extracting inner CAMUS archive...
CAMUS DATASET CHECK

Dataset path:
/content/camus_data_final/database_nifti

Patient directories found: 500

✓ All 500 CAMUS patient directories found.


In [ ]:
# ================================================================
# PHASE 3C — CELL 8
# VERIFY REQUIRED CAMUS A4C FILES — CORRECTED
# ================================================================

missing_records = []


for patient_dir in patient_dirs:

    patient_id = patient_dir.name


    # ------------------------------------------------------------
    # IMAGE / MASK FILES
    # These contain the patient ID in the filename
    # ------------------------------------------------------------

    required_patient_files = [

        f"{patient_id}_4CH_ED.nii.gz",

        f"{patient_id}_4CH_ES.nii.gz",

        f"{patient_id}_4CH_ED_gt.nii.gz",

        f"{patient_id}_4CH_ES_gt.nii.gz"
    ]


    for filename in required_patient_files:

        expected_path = (
            patient_dir /
            filename
        )

        if not expected_path.exists():

            missing_records.append({

                "patient_id":
                    patient_id,

                "missing_file":
                    filename
            })


    # ------------------------------------------------------------
    # CAMUS CONFIGURATION FILE
    # IMPORTANT:
    # Filename is Info_4CH.cfg — patient ID is NOT prefixed
    # ------------------------------------------------------------

    info_path = (
        patient_dir /
        "Info_4CH.cfg"
    )


    if not info_path.exists():

        missing_records.append({

            "patient_id":
                patient_id,

            "missing_file":
                "Info_4CH.cfg"
        })


# ================================================================
# REPORT
# ================================================================

print("=" * 80)
print("CAMUS A4C FILE VERIFICATION")
print("=" * 80)


print(
    "\nPatients checked:",
    len(patient_dirs)
)


print(
    "Missing required files:",
    len(missing_records)
)


# ---------------------------------------------------------------
# FAIL SAFELY IF SOMETHING REALLY IS MISSING
# ---------------------------------------------------------------

if len(missing_records) > 0:

    missing_df = pd.DataFrame(
        missing_records
    )

    print(
        "\nMissing-file examples:"
    )

    display(
        missing_df.head(30)
    )


    print(
        "\nMissing-file types:"
    )

    display(
        missing_df[
            "missing_file"
        ]
        .value_counts()
        .rename_axis(
            "filename"
        )
        .reset_index(
            name="count"
        )
    )


    raise FileNotFoundError(
        f"{len(missing_records)} required "
        "CAMUS files are missing."
    )


# ================================================================
# SUCCESS
# ================================================================

print(
    "\n✓ ED images found for all 500 patients."
)

print(
    "✓ ES images found for all 500 patients."
)

print(
    "✓ ED reference masks found for all 500 patients."
)

print(
    "✓ ES reference masks found for all 500 patients."
)

print(
    "✓ Info_4CH.cfg found for all 500 patients."
)


print(
    "\n✓ CAMUS A4C dataset integrity check passed."
)

CAMUS A4C FILE VERIFICATION

Patients checked: 500
Missing required files: 0

✓ ED images found for all 500 patients.
✓ ES images found for all 500 patients.
✓ ED reference masks found for all 500 patients.
✓ ES reference masks found for all 500 patients.
✓ Info_4CH.cfg found for all 500 patients.

✓ CAMUS A4C dataset integrity check passed.


In [ ]:
# ================================================================
# PHASE 3C — CELL 9
# LOAD AND VERIFY FROZEN LV SEGMENTATION MODEL
# ================================================================

# ---------------------------------------------------------------
# SHA-256 HELPER
# ---------------------------------------------------------------

def calculate_sha256(
    file_path,
    chunk_size=1024 * 1024
):

    sha256 = hashlib.sha256()

    with open(
        file_path,
        "rb"
    ) as file:

        while True:

            chunk = file.read(
                chunk_size
            )

            if not chunk:
                break

            sha256.update(
                chunk
            )

    return sha256.hexdigest()


# ---------------------------------------------------------------
# EXPECTED FROZEN HASH
# ---------------------------------------------------------------

EXPECTED_SEGMENTATION_SHA256 = (
    "65ec53959831c6b7c08747a5d94de656"
    "3580ce9ec052aa7b9afa614bee5705d3"
)


actual_segmentation_sha256 = (
    calculate_sha256(
        FINAL_SEG_MODEL_PATH
    )
)


print("=" * 80)
print("FROZEN SEGMENTATION MODEL VERIFICATION")
print("=" * 80)

print(
    "\nExpected SHA-256:"
)

print(
    EXPECTED_SEGMENTATION_SHA256
)

print(
    "\nActual SHA-256:"
)

print(
    actual_segmentation_sha256
)


assert (
    actual_segmentation_sha256
    ==
    EXPECTED_SEGMENTATION_SHA256
), (
    "Segmentation model hash does not match "
    "the frozen Phase 1F model."
)


print(
    "\n✓ SHA-256 matches frozen Phase 1F model."
)


# ---------------------------------------------------------------
# LOAD MODEL
# ---------------------------------------------------------------

frozen_segmentation_model = (
    tf.keras.models.load_model(
        FINAL_SEG_MODEL_PATH,
        compile=False
    )
)


frozen_segmentation_model.trainable = False


print(
    "\nModel input shape :",
    frozen_segmentation_model.input_shape
)

print(
    "Model output shape:",
    frozen_segmentation_model.output_shape
)

print(
    "Model parameters  :",
    f"{frozen_segmentation_model.count_params():,}"
)

print(
    "Trainable         :",
    frozen_segmentation_model.trainable
)


# ---------------------------------------------------------------
# HARD MODEL CHECKS
# ---------------------------------------------------------------

assert (
    frozen_segmentation_model.input_shape
    ==
    (None, 256, 256, 1)
), (
    "Unexpected model input shape."
)


assert (
    frozen_segmentation_model.output_shape
    ==
    (None, 256, 256, 1)
), (
    "Unexpected model output shape."
)


assert (
    frozen_segmentation_model.count_params()
    ==
    7_759_521
), (
    "Unexpected segmentation model parameter count."
)


print(
    "\n✓ Frozen segmentation model loaded successfully."
)

print(
    "✓ Model remains non-trainable during Phase 3C."
)

FROZEN SEGMENTATION MODEL VERIFICATION

Expected SHA-256:
65ec53959831c6b7c08747a5d94de6563580ce9ec052aa7b9afa614bee5705d3

Actual SHA-256:
65ec53959831c6b7c08747a5d94de6563580ce9ec052aa7b9afa614bee5705d3

✓ SHA-256 matches frozen Phase 1F model.

Model input shape : (None, 256, 256, 1)
Model output shape: (None, 256, 256, 1)
Model parameters  : 7,759,521
Trainable         : False

✓ Frozen segmentation model loaded successfully.
✓ Model remains non-trainable during Phase 3C.


In [ ]:
# ================================================================
# PHASE 3C — CELL 10
# EXACT FROZEN PHASE 2B CAMUS LOADER
# ================================================================

def parse_info_cfg(cfg_path):

    info = {}

    with open(
        cfg_path,
        "r",
        encoding="utf-8"
    ) as f:

        for raw in f:

            line = raw.strip()

            if (
                not line
                or line.startswith("#")
                or ":" not in line
            ):
                continue

            key, value = line.split(
                ":",
                1
            )

            key = key.strip()
            value = value.strip()

            try:
                value = int(value)

            except ValueError:

                try:
                    value = float(value)

                except ValueError:
                    pass

            info[key] = value

    return info


def load_nifti_2d(file_path):

    nii = nib.load(
        str(file_path)
    )

    data = np.asarray(
        nii.get_fdata()
    )

    if (
        data.ndim == 3
        and data.shape[-1] == 1
    ):
        data = data[:, :, 0]

    if data.ndim != 2:

        raise ValueError(
            f"Expected 2D NIfTI, got {data.shape}"
        )


    spacing = tuple(
        float(x)
        for x in
        nii.header.get_zooms()[:2]
    )


    return data, spacing


def load_patient_data(patient_id):

    patient_dir = (
        DATASET_PATH /
        patient_id
    )

    clinical = parse_info_cfg(
        patient_dir /
        "Info_4CH.cfg"
    )


    patient = {

        "patient_id":
            patient_id,

        "clinical":
            clinical
    }


    for phase in ["ED", "ES"]:

        image, spacing = (
            load_nifti_2d(
                patient_dir /
                f"{patient_id}_4CH_{phase}.nii.gz"
            )
        )

        gt, gt_spacing = (
            load_nifti_2d(
                patient_dir /
                f"{patient_id}_4CH_{phase}_gt.nii.gz"
            )
        )


        assert np.allclose(
            spacing,
            gt_spacing
        )


        patient[phase] = {

            "image":
                image.astype(
                    np.float32
                ),

            "mask":
                (gt == 1),

            "spacing":
                spacing,

            "original_shape":
                image.shape
        }


    return patient


print("✓ Exact frozen Phase 2B CAMUS loader ready.")

✓ Exact frozen Phase 2B CAMUS loader ready.


In [ ]:
# ================================================================
# PHASE 3C — CELL 11
# EXACT FROZEN PHASE 2B PREPROCESSING
# ================================================================

def preprocess_for_segmentation(
    image,
    target_size=256
):

    image = np.asarray(
        image,
        dtype=np.float32
    )

    original_h, original_w = (
        image.shape
    )


    # IMPORTANT:
    # Frozen Phase 2B resizes FIRST.
    resized = cv2.resize(
        image,
        (
            target_size,
            target_size
        ),
        interpolation=cv2.INTER_LINEAR
    )


    # Then performs min-max normalization.
    minimum = float(
        resized.min()
    )

    maximum = float(
        resized.max()
    )


    if maximum > minimum:

        normalized = (
            resized - minimum
        ) / (
            maximum - minimum
        )

    else:

        normalized = (
            np.zeros_like(
                resized
            )
        )


    scale_x = (
        original_w /
        target_size
    )

    scale_y = (
        original_h /
        target_size
    )


    return (
        normalized.astype(
            np.float32
        ),
        (
            scale_x,
            scale_y
        )
    )


def preprocess_mask(
    mask,
    target_size=256
):

    resized = cv2.resize(
        np.asarray(mask).astype(
            np.uint8
        ),
        (
            target_size,
            target_size
        ),
        interpolation=cv2.INTER_NEAREST
    )

    return resized > 0


def effective_spacing(
    native_spacing,
    scale_factors
):

    spacing_x, spacing_y = map(
        float,
        native_spacing
    )

    scale_x, scale_y = map(
        float,
        scale_factors
    )


    return (
        spacing_x * scale_x,
        spacing_y * scale_y
    )


print("✓ Exact frozen Phase 2B preprocessing ready.")

✓ Exact frozen Phase 2B preprocessing ready.


In [ ]:
# ================================================================
# PHASE 3C — CELL 12
# EXACT FROZEN PHASE 2B POST-PROCESSING
# ================================================================

def keep_largest_component(mask):

    mask_uint8 = (
        np.asarray(mask)
        .astype(np.uint8)
    )


    if mask_uint8.sum() == 0:

        return np.zeros_like(
            mask_uint8,
            dtype=bool
        )


    n_labels, labels, stats, _ = (
        cv2.connectedComponentsWithStats(
            mask_uint8,
            connectivity=8
        )
    )


    if n_labels <= 1:

        return np.zeros_like(
            mask_uint8,
            dtype=bool
        )


    largest_label = (
        1
        +
        np.argmax(
            stats[
                1:,
                cv2.CC_STAT_AREA
            ]
        )
    )


    return labels == largest_label


def postprocess_prediction(
    probability_mask,
    threshold=0.50
):

    binary_mask = (
        probability_mask
        >=
        threshold
    )


    largest = (
        keep_largest_component(
            binary_mask
        )
    )


    filled = (
        binary_fill_holes(
            largest
        )
    )


    return np.asarray(
        filled
    ).astype(bool)


print("✓ Exact frozen Phase 2B post-processing ready.")

✓ Exact frozen Phase 2B post-processing ready.


In [ ]:
# ================================================================
# PHASE 3C — CELL 13
# EXACT FROZEN PHASE 2B QUANTITATIVE GEOMETRY
# ================================================================

def mask_area_mm2(
    mask,
    spacing
):

    mask = np.asarray(
        mask
    ).astype(bool)

    spacing_x, spacing_y = map(
        float,
        spacing
    )


    return float(
        mask.sum()
        *
        spacing_x
        *
        spacing_y
    )


def mask_points_mm(
    mask,
    spacing
):

    rows, cols = np.where(
        np.asarray(mask).astype(bool)
    )


    spacing_x, spacing_y = map(
        float,
        spacing
    )


    x_mm = (
        cols.astype(np.float64)
        *
        spacing_x
    )

    y_mm = (
        rows.astype(np.float64)
        *
        spacing_y
    )


    return np.column_stack([
        x_mm,
        y_mm
    ])


def pca_major_axis_mm(
    mask,
    spacing
):

    points = mask_points_mm(
        mask,
        spacing
    )


    if len(points) < 2:
        return np.nan


    centered = (
        points
        -
        points.mean(axis=0)
    )


    covariance = np.cov(
        centered,
        rowvar=False
    )


    eigenvalues, eigenvectors = (
        np.linalg.eigh(
            covariance
        )
    )


    principal_axis = (
        eigenvectors[
            :,
            np.argmax(
                eigenvalues
            )
        ]
    )


    projections = (
        centered
        @
        principal_axis
    )


    return float(
        projections.max()
        -
        projections.min()
    )


def area_length_volume_ml(
    area_mm2,
    length_mm
):

    if (
        not np.isfinite(area_mm2)
        or not np.isfinite(length_mm)
        or area_mm2 <= 0
        or length_mm <= 0
    ):
        return np.nan


    area_cm2 = (
        area_mm2 /
        100.0
    )

    length_cm = (
        length_mm /
        10.0
    )


    return float(

        (
            8.0
            /
            (
                3.0
                *
                np.pi
            )
        )

        *

        (
            area_cm2 ** 2
        )

        /

        length_cm
    )


print("✓ Exact frozen Phase 2B PCA area-length geometry ready.")

✓ Exact frozen Phase 2B PCA area-length geometry ready.


In [ ]:
# ================================================================
# PHASE 3C — CELL 14
# CRITICAL FROZEN PHASE 2B REPRODUCTION TEST
# ================================================================

VERIFICATION_PATIENT = "patient0173"


# ---------------------------------------------------------------
# LOAD ORIGINAL FROZEN PHASE 2B TABLE
# ---------------------------------------------------------------

frozen_phase2B_df = pd.read_csv(
    PHASE2_TEST_TABLE_PATH
)


assert (
    VERIFICATION_PATIENT
    in
    frozen_phase2B_df[
        "patient_id"
    ].values
), (
    f"{VERIFICATION_PATIENT} not found "
    "in frozen Phase 2B table."
)


frozen_row = (
    frozen_phase2B_df[
        frozen_phase2B_df[
            "patient_id"
        ]
        ==
        VERIFICATION_PATIENT
    ]
    .iloc[0]
)


# ---------------------------------------------------------------
# LOAD PATIENT USING EXACT PHASE 2B LOADER
# ---------------------------------------------------------------

patient_data = (
    load_patient_data(
        VERIFICATION_PATIENT
    )
)


recomputed = {}


# ---------------------------------------------------------------
# ED + ES
# ---------------------------------------------------------------

for phase in ["ED", "ES"]:

    phase_data = (
        patient_data[phase]
    )


    image = (
        phase_data[
            "image"
        ]
    )

    native_spacing = (
        phase_data[
            "spacing"
        ]
    )


    # ============================================================
    # EXACT PHASE 2B PREPROCESSING
    # ============================================================

    processed_image, scale_factors = (
        preprocess_for_segmentation(
            image,
            target_size=256
        )
    )


    resized_spacing = (
        effective_spacing(
            native_spacing,
            scale_factors
        )
    )


    # ============================================================
    # EXACT PHASE 2B MODEL INFERENCE
    # ============================================================

    model_input = (
        processed_image[
            None,
            ...,
            None
        ]
    )


    probability = (
        frozen_segmentation_model(
            model_input,
            training=False
        )
        .numpy()[0, :, :, 0]
    )


    predicted_mask = (
        postprocess_prediction(
            probability,
            threshold=0.50
        )
    )


    # ============================================================
    # EXACT PHASE 2B GEOMETRY
    # ============================================================

    pred_area = (
        mask_area_mm2(
            predicted_mask,
            resized_spacing
        )
    )


    pred_length = (
        pca_major_axis_mm(
            predicted_mask,
            resized_spacing
        )
    )


    pred_volume = (
        area_length_volume_ml(
            pred_area,
            pred_length
        )
    )


    recomputed[
        f"{phase}_PRED_area_mm2"
    ] = pred_area


    recomputed[
        f"{phase}_PRED_length_mm"
    ] = pred_length


    recomputed[
        f"{phase}_PRED_volume_ml"
    ] = pred_volume


    recomputed[
        f"{phase}_PRED_pixels"
    ] = int(
        predicted_mask.sum()
    )


# ---------------------------------------------------------------
# SV
# ---------------------------------------------------------------

recomputed[
    "PRED_SV_ml"
] = (
    recomputed[
        "ED_PRED_volume_ml"
    ]
    -
    recomputed[
        "ES_PRED_volume_ml"
    ]
)


# ---------------------------------------------------------------
# EF
# ---------------------------------------------------------------

recomputed[
    "PRED_EF_pct"
] = (
    recomputed[
        "PRED_SV_ml"
    ]
    /
    recomputed[
        "ED_PRED_volume_ml"
    ]
    *
    100.0
)


# ---------------------------------------------------------------
# COMPARE AGAINST FROZEN PHASE 2B
# ---------------------------------------------------------------

comparison_features = [

    "ED_PRED_area_mm2",
    "ED_PRED_length_mm",
    "ED_PRED_volume_ml",

    "ES_PRED_area_mm2",
    "ES_PRED_length_mm",
    "ES_PRED_volume_ml",

    "PRED_SV_ml",
    "PRED_EF_pct"
]


comparison_records = []


for feature in comparison_features:

    frozen_value = float(
        frozen_row[
            feature
        ]
    )

    recomputed_value = float(
        recomputed[
            feature
        ]
    )


    comparison_records.append({

        "feature":
            feature,

        "frozen_phase2B":
            frozen_value,

        "recomputed":
            recomputed_value,

        "absolute_difference":
            abs(
                frozen_value
                -
                recomputed_value
            )
    })


comparison_df = pd.DataFrame(
    comparison_records
)


print("=" * 80)
print("CRITICAL PHASE 2B REPRODUCTION TEST")
print("=" * 80)


print(
    "\nVerification patient:",
    VERIFICATION_PATIENT
)


display(
    comparison_df
)


print(
    "\nRecomputed predicted-mask pixels:"
)

print(
    "ED:",
    recomputed[
        "ED_PRED_pixels"
    ]
)

print(
    "ES:",
    recomputed[
        "ES_PRED_pixels"
    ]
)


max_difference = float(
    comparison_df[
        "absolute_difference"
    ].max()
)


print(
    "\nMaximum absolute difference:",
    f"{max_difference:.12f}"
)


# ---------------------------------------------------------------
# HARD GATE
# ---------------------------------------------------------------

TOLERANCE = 1e-6


if max_difference <= TOLERANCE:

    print(
        "\n✓ EXACT REPRODUCTION PASSED."
    )

    print(
        "✓ Current Phase 3C quantitative pipeline "
        "matches frozen Phase 2B."
    )

    print(
        "✓ Safe to proceed to train/validation "
        "feature generation."
    )

else:

    print(
        "\n✗ REPRODUCTION CHECK FAILED."
    )

    print(
        "✗ Do NOT generate Phase 3C features."
    )

    raise RuntimeError(
        "Frozen Phase 2B implementation was "
        "not reproduced within tolerance."
    )

CRITICAL PHASE 2B REPRODUCTION TEST

Verification patient: patient0173


,feature,frozen_phase2B,recomputed,absolute_difference
0,ED_PRED_area_mm2,2714.265708,2714.265708,0.000000e+00
1,ED_PRED_length_mm,78.143962,78.143962,0.000000e+00
2,ED_PRED_volume_ml,80.025456,80.025456,0.000000e+00
3,ES_PRED_area_mm2,1520.238767,1520.238767,0.000000e+00
4,ES_PRED_length_mm,62.295982,62.295982,0.000000e+00
5,ES_PRED_volume_ml,31.490708,31.490708,0.000000e+00
6,PRED_SV_ml,48.534747,48.534747,7.105427e-15
7,PRED_EF_pct,60.649136,60.649136,0.000000e+00



Recomputed predicted-mask pixels:
ED: 6515
ES: 3649

Maximum absolute difference: 0.000000000000

✓ EXACT REPRODUCTION PASSED.
✓ Current Phase 3C quantitative pipeline matches frozen Phase 2B.
✓ Safe to proceed to train/validation feature generation.


In [ ]:
# ================================================================
# PHASE 3C — CELL 15
# QUANTITATIVE FEATURE EXTRACTION FUNCTION
# ================================================================

def extract_phase3C_features(patient_id):

    patient_data = load_patient_data(
        patient_id
    )

    features = {
        "patient_id": patient_id
    }


    # ------------------------------------------------------------
    # PROCESS ED AND ES
    # ------------------------------------------------------------

    for phase in ["ED", "ES"]:

        phase_data = patient_data[
            phase
        ]


        # --------------------------------------------------------
        # PREPROCESS IMAGE
        # --------------------------------------------------------

        processed_image, scale_factors = (
            preprocess_for_segmentation(
                phase_data["image"],
                target_size=256
            )
        )


        resized_spacing = (
            effective_spacing(
                phase_data["spacing"],
                scale_factors
            )
        )


        # --------------------------------------------------------
        # FROZEN SEGMENTATION INFERENCE
        # --------------------------------------------------------

        model_input = (
            processed_image[
                None,
                ...,
                None
            ]
        )


        probability = (
            frozen_segmentation_model(
                model_input,
                training=False
            )
            .numpy()[0, :, :, 0]
        )


        predicted_mask = (
            postprocess_prediction(
                probability,
                threshold=0.50
            )
        )


        # --------------------------------------------------------
        # QUANTITATIVE GEOMETRY
        # --------------------------------------------------------

        area_mm2 = (
            mask_area_mm2(
                predicted_mask,
                resized_spacing
            )
        )


        length_mm = (
            pca_major_axis_mm(
                predicted_mask,
                resized_spacing
            )
        )


        volume_ml = (
            area_length_volume_ml(
                area_mm2,
                length_mm
            )
        )


        # --------------------------------------------------------
        # STORE PHASE FEATURES
        # --------------------------------------------------------

        features[
            f"PRED_{phase}_area_mm2"
        ] = area_mm2


        features[
            f"PRED_{phase}_length_mm"
        ] = length_mm


        features[
            f"PRED_{phase}V_ml"
        ] = volume_ml


    # ------------------------------------------------------------
    # STROKE VOLUME
    # ------------------------------------------------------------

    features[
        "PRED_SV_ml"
    ] = (
        features[
            "PRED_EDV_ml"
        ]
        -
        features[
            "PRED_ESV_ml"
        ]
    )


    # ------------------------------------------------------------
    # EJECTION FRACTION
    # ------------------------------------------------------------

    if (
        np.isfinite(
            features[
                "PRED_EDV_ml"
            ]
        )
        and
        features[
            "PRED_EDV_ml"
        ] > 0
    ):

        features[
            "PRED_EF_pct"
        ] = (
            features[
                "PRED_SV_ml"
            ]
            /
            features[
                "PRED_EDV_ml"
            ]
            *
            100.0
        )

    else:

        features[
            "PRED_EF_pct"
        ] = np.nan


    return features


print(
    "✓ Phase 3C quantitative feature "
    "extraction function ready."
)

✓ Phase 3C quantitative feature extraction function ready.


In [ ]:
# ================================================================
# PHASE 3C — CELL 16
# GENERATE TRAIN + VALIDATION QUANTITATIVE FEATURES
# ================================================================

FEATURE_CHECKPOINT_PATH = (
    PHASE3C_TABLE_DIR /
    "phase3C_quantitative_features_checkpoint.csv"
)

FINAL_FEATURE_TABLE_PATH = (
    PHASE3C_TABLE_DIR /
    "phase3C_quantitative_features_train_validation.csv"
)


# ---------------------------------------------------------------
# DEVELOPMENT PATIENTS ONLY
# ---------------------------------------------------------------

development_labels_df = (
    phase3C_labels_df
    .copy()
    .reset_index(drop=True)
)


assert len(
    development_labels_df
) == 450


assert set(
    development_labels_df[
        "split"
    ].unique()
) == {
    "train",
    "validation"
}


assert not (
    development_labels_df[
        "split"
    ]
    ==
    "test"
).any()


development_patient_ids = (
    development_labels_df[
        "patient_id"
    ]
    .tolist()
)


print("=" * 80)
print("PHASE 3C FEATURE GENERATION")
print("=" * 80)

print(
    "\nDevelopment patients:",
    len(
        development_patient_ids
    )
)

print(
    "Training:",
    int(
        (
            development_labels_df[
                "split"
            ]
            ==
            "train"
        ).sum()
    )
)

print(
    "Validation:",
    int(
        (
            development_labels_df[
                "split"
            ]
            ==
            "validation"
        ).sum()
    )
)

print(
    "Test:",
    int(
        (
            development_labels_df[
                "split"
            ]
            ==
            "test"
        ).sum()
    )
)


# ---------------------------------------------------------------
# SEQUENTIAL EXTRACTION
# ---------------------------------------------------------------

feature_records = []


for index, patient_id in enumerate(
    development_patient_ids,
    start=1
):

    try:

        patient_features = (
            extract_phase3C_features(
                patient_id
            )
        )


        feature_records.append(
            patient_features
        )


    except Exception as error:

        print(
            f"\n✗ ERROR for {patient_id}: "
            f"{error}"
        )

        raise


    # -----------------------------------------------------------
    # CHECKPOINT EVERY 25 PATIENTS
    # -----------------------------------------------------------

    if (
        index % 25 == 0
        or
        index == len(
            development_patient_ids
        )
    ):

        checkpoint_df = (
            pd.DataFrame(
                feature_records
            )
        )


        checkpoint_df.to_csv(
            FEATURE_CHECKPOINT_PATH,
            index=False
        )


        print(
            f"Processed "
            f"{index:>3}/"
            f"{len(development_patient_ids)} "
            f"patients"
        )


# ---------------------------------------------------------------
# CREATE FEATURE TABLE
# ---------------------------------------------------------------

phase3C_features_df = (
    pd.DataFrame(
        feature_records
    )
)


assert len(
    phase3C_features_df
) == 450


assert (
    phase3C_features_df[
        "patient_id"
    ]
    .nunique()
    ==
    450
)


# ---------------------------------------------------------------
# MERGE FROZEN LABELS
# ---------------------------------------------------------------

phase3C_features_df = (
    development_labels_df[
        [
            "patient_id",
            "split",
            "reference_EF_pct",
            "reference_EF_class"
        ]
    ]
    .merge(
        phase3C_features_df,
        on="patient_id",
        how="left",
        validate="one_to_one"
    )
)


# ---------------------------------------------------------------
# SAVE FINAL DEVELOPMENT FEATURE TABLE
# ---------------------------------------------------------------

phase3C_features_df.to_csv(
    FINAL_FEATURE_TABLE_PATH,
    index=False
)


print("\n" + "=" * 80)

print(
    "✓ Feature generation complete."
)

print(
    "✓ Patients:",
    len(
        phase3C_features_df
    )
)

print(
    "✓ Test patients used: 0"
)

print(
    "\nSaved:"
)

print(
    FINAL_FEATURE_TABLE_PATH
)

PHASE 3C FEATURE GENERATION

Development patients: 450
Training: 400
Validation: 50
Test: 0
Processed  25/450 patients
Processed  50/450 patients
Processed  75/450 patients
Processed 100/450 patients
Processed 125/450 patients
Processed 150/450 patients
Processed 175/450 patients
Processed 200/450 patients
Processed 225/450 patients
Processed 250/450 patients
Processed 275/450 patients
Processed 300/450 patients
Processed 325/450 patients
Processed 350/450 patients
Processed 375/450 patients
Processed 400/450 patients
Processed 425/450 patients
Processed 450/450 patients

✓ Feature generation complete.
✓ Patients: 450
✓ Test patients used: 0

Saved:
/content/drive/MyDrive/CAMUS_FINAL/phase3_classification/phase3C_classifier_development/tables/phase3C_quantitative_features_train_validation.csv


In [ ]:
# ================================================================
# PHASE 3C — CELL 17
# FEATURE TABLE INTEGRITY + SANITY CHECKS
# ================================================================

QUANTITATIVE_FEATURES = [

    "PRED_ED_area_mm2",
    "PRED_ES_area_mm2",

    "PRED_ED_length_mm",
    "PRED_ES_length_mm",

    "PRED_EDV_ml",
    "PRED_ESV_ml",

    "PRED_SV_ml",
    "PRED_EF_pct"
]


print("=" * 80)
print("PHASE 3C FEATURE INTEGRITY CHECK")
print("=" * 80)


# ---------------------------------------------------------------
# BASIC TABLE CHECKS
# ---------------------------------------------------------------

print(
    "\nTable shape:",
    phase3C_features_df.shape
)


print(
    "\nSplit counts:"
)

print(
    phase3C_features_df[
        "split"
    ]
    .value_counts()
    .to_string()
)


assert len(
    phase3C_features_df
) == 450


assert (
    phase3C_features_df[
        "patient_id"
    ]
    .nunique()
    ==
    450
)


assert not (
    phase3C_features_df[
        "split"
    ]
    ==
    "test"
).any()


# ---------------------------------------------------------------
# MISSING / NON-FINITE VALUES
# ---------------------------------------------------------------

missing_counts = (
    phase3C_features_df[
        QUANTITATIVE_FEATURES
    ]
    .isna()
    .sum()
)


nonfinite_counts = {}


for feature in QUANTITATIVE_FEATURES:

    values = (
        phase3C_features_df[
            feature
        ]
        .to_numpy(
            dtype=float
        )
    )

    nonfinite_counts[
        feature
    ] = int(
        (
            ~np.isfinite(
                values
            )
        ).sum()
    )


integrity_df = pd.DataFrame({

    "feature":
        QUANTITATIVE_FEATURES,

    "missing":
        [
            int(
                missing_counts[
                    feature
                ]
            )
            for feature
            in QUANTITATIVE_FEATURES
        ],

    "nonfinite":
        [
            nonfinite_counts[
                feature
            ]
            for feature
            in QUANTITATIVE_FEATURES
        ],

    "minimum":
        [
            float(
                phase3C_features_df[
                    feature
                ].min()
            )
            for feature
            in QUANTITATIVE_FEATURES
        ],

    "maximum":
        [
            float(
                phase3C_features_df[
                    feature
                ].max()
            )
            for feature
            in QUANTITATIVE_FEATURES
        ]
})


print(
    "\nFeature integrity:"
)

display(
    integrity_df
)


# ---------------------------------------------------------------
# PHYSIOLOGICAL / MATHEMATICAL CHECKS
# ---------------------------------------------------------------

check_results = {

    "ED area > 0":
        int(
            (
                phase3C_features_df[
                    "PRED_ED_area_mm2"
                ]
                <= 0
            ).sum()
        ),

    "ES area > 0":
        int(
            (
                phase3C_features_df[
                    "PRED_ES_area_mm2"
                ]
                <= 0
            ).sum()
        ),

    "ED length > 0":
        int(
            (
                phase3C_features_df[
                    "PRED_ED_length_mm"
                ]
                <= 0
            ).sum()
        ),

    "ES length > 0":
        int(
            (
                phase3C_features_df[
                    "PRED_ES_length_mm"
                ]
                <= 0
            ).sum()
        ),

    "EDV > 0":
        int(
            (
                phase3C_features_df[
                    "PRED_EDV_ml"
                ]
                <= 0
            ).sum()
        ),

    "ESV > 0":
        int(
            (
                phase3C_features_df[
                    "PRED_ESV_ml"
                ]
                <= 0
            ).sum()
        ),

    "EDV > ESV":
        int(
            (
                phase3C_features_df[
                    "PRED_EDV_ml"
                ]
                <=
                phase3C_features_df[
                    "PRED_ESV_ml"
                ]
            ).sum()
        ),

    "EF within 0–100":
        int(
            (
                (
                    phase3C_features_df[
                        "PRED_EF_pct"
                    ]
                    <
                    0
                )
                |
                (
                    phase3C_features_df[
                        "PRED_EF_pct"
                    ]
                    >
                    100
                )
            ).sum()
        )
}


sanity_df = pd.DataFrame({

    "check":
        list(
            check_results.keys()
        ),

    "failed_patients":
        list(
            check_results.values()
        )
})


print(
    "\nSanity checks:"
)

display(
    sanity_df
)


# ---------------------------------------------------------------
# HARD INTEGRITY GATE
# ---------------------------------------------------------------

total_missing = int(
    integrity_df[
        "missing"
    ].sum()
)


total_nonfinite = int(
    integrity_df[
        "nonfinite"
    ].sum()
)


total_failed_checks = int(
    sanity_df[
        "failed_patients"
    ].sum()
)


print("\n" + "=" * 80)

print(
    "Missing values      :",
    total_missing
)

print(
    "Non-finite values   :",
    total_nonfinite
)

print(
    "Failed sanity checks:",
    total_failed_checks
)


if (
    total_missing == 0
    and
    total_nonfinite == 0
    and
    total_failed_checks == 0
):

    print(
        "\n✓ FEATURE INTEGRITY CHECK PASSED."
    )

    print(
        "✓ 450 train/validation patients "
        "have valid quantitative features."
    )

    print(
        "✓ Safe to proceed to classifier development."
    )

else:

    print(
        "\n⚠ FEATURE INTEGRITY CHECK REQUIRES REVIEW."
    )

    print(
        "Do NOT train classifiers yet."
    )

PHASE 3C FEATURE INTEGRITY CHECK

Table shape: (450, 12)

Split counts:
split
train         400
validation     50

Feature integrity:


,feature,missing,nonfinite,minimum,maximum
0,PRED_ED_area_mm2,0,0,1333.737597,6136.873453
1,PRED_ES_area_mm2,0,0,716.564486,5970.279034
2,PRED_ED_length_mm,0,0,53.516805,110.455787
3,PRED_ES_length_mm,0,0,45.084752,107.344418
4,PRED_EDV_ml,0,0,26.188092,291.913960
5,PRED_ESV_ml,0,0,8.370590,281.856889
6,PRED_SV_ml,0,0,4.070053,110.962009
7,PRED_EF_pct,0,0,3.445218,85.905682



Sanity checks:


,check,failed_patients
0,ED area > 0,0
1,ES area > 0,0
2,ED length > 0,0
3,ES length > 0,0
4,EDV > 0,0
5,ESV > 0,0
6,EDV > ESV,0
7,EF within 0–100,0



Missing values      : 0
Non-finite values   : 0
Failed sanity checks: 0

✓ FEATURE INTEGRITY CHECK PASSED.
✓ 450 train/validation patients have valid quantitative features.
✓ Safe to proceed to classifier development.


In [ ]:
# ================================================================
# PHASE 3C — CELL 18
# PREPARE TRAIN / VALIDATION DATA
# ================================================================

FEATURE_SETS = {

    "F1_EF_only": [
        "PRED_EF_pct"
    ],

    "F2_volumetric": [
        "PRED_EDV_ml",
        "PRED_ESV_ml",
        "PRED_SV_ml",
        "PRED_EF_pct"
    ],

    "F3_full_quantitative": [
        "PRED_ED_area_mm2",
        "PRED_ES_area_mm2",
        "PRED_ED_length_mm",
        "PRED_ES_length_mm",
        "PRED_EDV_ml",
        "PRED_ESV_ml",
        "PRED_SV_ml",
        "PRED_EF_pct"
    ]
}


CLASS_ORDER = [
    "Lower_EF",
    "Intermediate_EF",
    "Higher_EF"
]


TARGET_COLUMN = (
    "reference_EF_class"
)


train_df = (
    phase3C_features_df[
        phase3C_features_df["split"]
        ==
        "train"
    ]
    .copy()
    .reset_index(drop=True)
)


validation_df = (
    phase3C_features_df[
        phase3C_features_df["split"]
        ==
        "validation"
    ]
    .copy()
    .reset_index(drop=True)
)


# ---------------------------------------------------------------
# HARD CHECKS
# ---------------------------------------------------------------

assert len(train_df) == 400
assert len(validation_df) == 50


assert (
    train_df["patient_id"]
    .isin(
        validation_df["patient_id"]
    )
    .sum()
    ==
    0
)


assert set(
    train_df[
        TARGET_COLUMN
    ].unique()
) == set(
    CLASS_ORDER
)


assert set(
    validation_df[
        TARGET_COLUMN
    ].unique()
) == set(
    CLASS_ORDER
)


print("=" * 80)
print("PHASE 3C DEVELOPMENT SPLIT")
print("=" * 80)


print(
    "\nTraining patients   :",
    len(train_df)
)

print(
    "Validation patients :",
    len(validation_df)
)

print(
    "Test patients       : 0"
)


print(
    "\nTraining class distribution:"
)

print(
    train_df[
        TARGET_COLUMN
    ]
    .value_counts()
    .reindex(
        CLASS_ORDER
    )
    .to_string()
)


print(
    "\nValidation class distribution:"
)

print(
    validation_df[
        TARGET_COLUMN
    ]
    .value_counts()
    .reindex(
        CLASS_ORDER
    )
    .to_string()
)


print(
    "\n✓ Train/validation development split ready."
)

PHASE 3C DEVELOPMENT SPLIT

Training patients   : 400
Validation patients : 50
Test patients       : 0

Training class distribution:
reference_EF_class
Lower_EF           143
Intermediate_EF    125
Higher_EF          132

Validation class distribution:
reference_EF_class
Lower_EF           13
Intermediate_EF    17
Higher_EF          20

✓ Train/validation development split ready.


In [ ]:
# ================================================================
# PHASE 3C — CELL 19
# TRAIN PREDEFINED LOGISTIC-REGRESSION CLASSIFIERS
# ================================================================

phase3C_models = {}

phase3C_validation_predictions = {}

phase3C_results = []


y_train = (
    train_df[
        TARGET_COLUMN
    ]
    .to_numpy()
)


y_validation = (
    validation_df[
        TARGET_COLUMN
    ]
    .to_numpy()
)


for feature_set_name, feature_columns in FEATURE_SETS.items():

    print(
        "\n" + "=" * 80
    )

    print(
        "Training:",
        feature_set_name
    )

    print(
        "=" * 80
    )


    # -----------------------------------------------------------
    # DATA
    # -----------------------------------------------------------

    X_train = (
        train_df[
            feature_columns
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    X_validation = (
        validation_df[
            feature_columns
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    # -----------------------------------------------------------
    # PIPELINE
    # Scaling is fitted ONLY on training data.
    # -----------------------------------------------------------

    classifier_pipeline = Pipeline([

        (
            "scaler",
            StandardScaler()
        ),

        (
            "classifier",
            LogisticRegression(
                solver="lbfgs",
                max_iter=5000,
                random_state=SEED
            )
        )
    ])


    # -----------------------------------------------------------
    # TRAIN
    # -----------------------------------------------------------

    classifier_pipeline.fit(
        X_train,
        y_train
    )


    # -----------------------------------------------------------
    # VALIDATION PREDICTIONS
    # -----------------------------------------------------------

    y_pred = (
        classifier_pipeline.predict(
            X_validation
        )
    )


    # -----------------------------------------------------------
    # METRICS
    # -----------------------------------------------------------

    accuracy = accuracy_score(
        y_validation,
        y_pred
    )


    balanced_accuracy = (
        balanced_accuracy_score(
            y_validation,
            y_pred
        )
    )


    macro_f1 = f1_score(
        y_validation,
        y_pred,
        average="macro",
        zero_division=0
    )


    weighted_f1 = f1_score(
        y_validation,
        y_pred,
        average="weighted",
        zero_division=0
    )


    kappa = cohen_kappa_score(
        y_validation,
        y_pred
    )


    # -----------------------------------------------------------
    # SAVE IN MEMORY
    # -----------------------------------------------------------

    phase3C_models[
        feature_set_name
    ] = classifier_pipeline


    phase3C_validation_predictions[
        feature_set_name
    ] = y_pred


    phase3C_results.append({

        "feature_set":
            feature_set_name,

        "n_features":
            len(feature_columns),

        "accuracy":
            accuracy,

        "balanced_accuracy":
            balanced_accuracy,

        "macro_f1":
            macro_f1,

        "weighted_f1":
            weighted_f1,

        "cohen_kappa":
            kappa
    })


    print(
        f"Features          : "
        f"{len(feature_columns)}"
    )

    print(
        f"Accuracy          : "
        f"{accuracy:.4f}"
    )

    print(
        f"Balanced Accuracy : "
        f"{balanced_accuracy:.4f}"
    )

    print(
        f"Macro F1          : "
        f"{macro_f1:.4f}"
    )

    print(
        f"Weighted F1       : "
        f"{weighted_f1:.4f}"
    )

    print(
        f"Cohen Kappa       : "
        f"{kappa:.4f}"
    )


print(
    "\n✓ All three predefined classifiers trained."
)

print(
    "✓ Selection information comes from validation only."
)


Training: F1_EF_only
Features          : 1
Accuracy          : 0.4200
Balanced Accuracy : 0.4394
Macro F1          : 0.4353
Weighted F1       : 0.4199
Cohen Kappa       : 0.1255

Training: F2_volumetric
Features          : 4
Accuracy          : 0.4800
Balanced Accuracy : 0.4983
Macro F1          : 0.4929
Weighted F1       : 0.4790
Cohen Kappa       : 0.2183

Training: F3_full_quantitative
Features          : 8
Accuracy          : 0.4800
Balanced Accuracy : 0.4983
Macro F1          : 0.4969
Weighted F1       : 0.4804
Cohen Kappa       : 0.2150

✓ All three predefined classifiers trained.
✓ Selection information comes from validation only.


In [ ]:
# ================================================================
# PHASE 3C — CELL 20
# VALIDATION COMPARISON + PREDEFINED MODEL SELECTION
# ================================================================

phase3C_results_df = (
    pd.DataFrame(
        phase3C_results
    )
)


# ---------------------------------------------------------------
# SORT FOR DISPLAY
# Primary metric = Macro F1
# ---------------------------------------------------------------

phase3C_results_df = (
    phase3C_results_df
    .sort_values(
        by=[
            "macro_f1",
            "balanced_accuracy",
            "accuracy"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)


print("=" * 80)
print("PHASE 3C — VALIDATION MODEL COMPARISON")
print("=" * 80)


display(
    phase3C_results_df.style.format({

        "accuracy":
            "{:.4f}",

        "balanced_accuracy":
            "{:.4f}",

        "macro_f1":
            "{:.4f}",

        "weighted_f1":
            "{:.4f}",

        "cohen_kappa":
            "{:.4f}"
    })
)


# ---------------------------------------------------------------
# SELECT USING PRIMARY METRIC
# ---------------------------------------------------------------

best_macro_f1 = (
    phase3C_results_df[
        "macro_f1"
    ].max()
)


macro_f1_winners = (
    phase3C_results_df[
        np.isclose(
            phase3C_results_df[
                "macro_f1"
            ],
            best_macro_f1
        )
    ]
    .copy()
)


# ---------------------------------------------------------------
# TIE HANDLING
#
# If exact Macro-F1 tie:
# prefer fewer features.
#
# This avoids silently selecting a more complex representation
# when validation performance is identical.
# ---------------------------------------------------------------

if len(
    macro_f1_winners
) > 1:

    print(
        "\nExact validation Macro-F1 tie detected."
    )

    print(
        "Applying conservative tie-break: "
        "fewer features."
    )

    selected_row = (
        macro_f1_winners
        .sort_values(
            by="n_features",
            ascending=True
        )
        .iloc[0]
    )

else:

    selected_row = (
        macro_f1_winners
        .iloc[0]
    )


SELECTED_FEATURE_SET = (
    selected_row[
        "feature_set"
    ]
)


SELECTED_FEATURES = (
    FEATURE_SETS[
        SELECTED_FEATURE_SET
    ]
)


selected_phase3C_model = (
    phase3C_models[
        SELECTED_FEATURE_SET
    ]
)


print("\n" + "=" * 80)

print(
    "SELECTED FEATURE SET:",
    SELECTED_FEATURE_SET
)

print(
    "Number of features :",
    len(
        SELECTED_FEATURES
    )
)

print(
    "Validation Macro F1:",
    f"{selected_row['macro_f1']:.4f}"
)


print(
    "\nSelected features:"
)

for feature in SELECTED_FEATURES:

    print(
        " •",
        feature
    )


print(
    "\n✓ Selection performed using validation data only."
)

print(
    "✓ Test partition remains unused."
)

PHASE 3C — VALIDATION MODEL COMPARISON


,feature_set,n_features,accuracy,balanced_accuracy,macro_f1,weighted_f1,cohen_kappa
0,F3_full_quantitative,8,0.4800,0.4983,0.4969,0.4804,0.2150
1,F2_volumetric,4,0.4800,0.4983,0.4929,0.4790,0.2183
2,F1_EF_only,1,0.4200,0.4394,0.4353,0.4199,0.1255



SELECTED FEATURE SET: F3_full_quantitative
Number of features : 8
Validation Macro F1: 0.4969

Selected features:
 • PRED_ED_area_mm2
 • PRED_ES_area_mm2
 • PRED_ED_length_mm
 • PRED_ES_length_mm
 • PRED_EDV_ml
 • PRED_ESV_ml
 • PRED_SV_ml
 • PRED_EF_pct

✓ Selection performed using validation data only.
✓ Test partition remains unused.


In [ ]:
# ================================================================
# PHASE 3C — CELL 21
# DETAILED VALIDATION EVALUATION OF SELECTED CLASSIFIER
# ================================================================

selected_validation_pred = (
    phase3C_validation_predictions[
        SELECTED_FEATURE_SET
    ]
)


# ---------------------------------------------------------------
# CONFUSION MATRIX
# ---------------------------------------------------------------

selected_confusion_matrix = (
    confusion_matrix(
        y_validation,
        selected_validation_pred,
        labels=CLASS_ORDER
    )
)


confusion_df = pd.DataFrame(

    selected_confusion_matrix,

    index=[
        f"True_{label}"
        for label in CLASS_ORDER
    ],

    columns=[
        f"Pred_{label}"
        for label in CLASS_ORDER
    ]
)


print("=" * 80)
print("SELECTED PHASE 3C CLASSIFIER — VALIDATION ANALYSIS")
print("=" * 80)


print(
    "\nSelected feature set:",
    SELECTED_FEATURE_SET
)


print(
    "\nConfusion matrix:"
)

display(
    confusion_df
)


# ---------------------------------------------------------------
# CLASSIFICATION REPORT
# ---------------------------------------------------------------

report_dict = (
    classification_report(
        y_validation,
        selected_validation_pred,
        labels=CLASS_ORDER,
        output_dict=True,
        zero_division=0
    )
)


per_class_records = []


for class_name in CLASS_ORDER:

    class_metrics = (
        report_dict[
            class_name
        ]
    )


    per_class_records.append({

        "class":
            class_name,

        "precision":
            class_metrics[
                "precision"
            ],

        "recall":
            class_metrics[
                "recall"
            ],

        "f1":
            class_metrics[
                "f1-score"
            ],

        "support":
            int(
                class_metrics[
                    "support"
                ]
            )
    })


per_class_df = pd.DataFrame(
    per_class_records
)


print(
    "\nPer-class validation performance:"
)


display(
    per_class_df.style.format({

        "precision":
            "{:.4f}",

        "recall":
            "{:.4f}",

        "f1":
            "{:.4f}"
    })
)


# ---------------------------------------------------------------
# MISCLASSIFIED PATIENTS
# ---------------------------------------------------------------

validation_analysis_df = (
    validation_df[
        [
            "patient_id",
            "reference_EF_pct",
            "reference_EF_class",
            "PRED_EF_pct"
        ]
    ]
    .copy()
)


validation_analysis_df[
    "classifier_prediction"
] = (
    selected_validation_pred
)


validation_analysis_df[
    "correct"
] = (
    validation_analysis_df[
        "reference_EF_class"
    ]
    ==
    validation_analysis_df[
        "classifier_prediction"
    ]
)


validation_analysis_df[
    "absolute_EF_error_pp"
] = (

    validation_analysis_df[
        "PRED_EF_pct"
    ]

    -

    validation_analysis_df[
        "reference_EF_pct"
    ]

).abs()


misclassified_validation_df = (

    validation_analysis_df[
        ~validation_analysis_df[
            "correct"
        ]
    ]

    .sort_values(
        by="absolute_EF_error_pp",
        ascending=False
    )

    .reset_index(drop=True)
)


print(
    "\nCorrect classifications:",
    int(
        validation_analysis_df[
            "correct"
        ].sum()
    ),
    "/",
    len(
        validation_analysis_df
    )
)


print(
    "Misclassifications:",
    len(
        misclassified_validation_df
    )
)


print(
    "\nMisclassified validation patients:"
)


display(
    misclassified_validation_df
)


# ---------------------------------------------------------------
# SAVE DEVELOPMENT RESULTS
# ---------------------------------------------------------------

RESULTS_PATH = (
    PHASE3C_TABLE_DIR /
    "phase3C_validation_model_comparison.csv"
)


PER_CLASS_PATH = (
    PHASE3C_TABLE_DIR /
    "phase3C_selected_validation_per_class.csv"
)


MISCLASSIFIED_PATH = (
    PHASE3C_TABLE_DIR /
    "phase3C_validation_misclassifications.csv"
)


phase3C_results_df.to_csv(
    RESULTS_PATH,
    index=False
)


per_class_df.to_csv(
    PER_CLASS_PATH,
    index=False
)


misclassified_validation_df.to_csv(
    MISCLASSIFIED_PATH,
    index=False
)


print(
    "\n✓ Validation analysis saved."
)

print(
    "✓ Test partition has still not been used."
)

SELECTED PHASE 3C CLASSIFIER — VALIDATION ANALYSIS

Selected feature set: F3_full_quantitative

Confusion matrix:


,Pred_Lower_EF,Pred_Intermediate_EF,Pred_Higher_EF
True_Lower_EF,8,4,1
True_Intermediate_EF,2,9,6
True_Higher_EF,2,11,7



Per-class validation performance:


,class,precision,recall,f1,support
0,Lower_EF,0.6667,0.6154,0.6400,13
1,Intermediate_EF,0.3750,0.5294,0.4390,17
2,Higher_EF,0.5000,0.3500,0.4118,20



Correct classifications: 24 / 50
Misclassifications: 26

Misclassified validation patients:


,patient_id,reference_EF_pct,reference_EF_class,PRED_EF_pct,classifier_prediction,correct,absolute_EF_error_pp
0,patient0099,30.0,Lower_EF,58.946022,Higher_EF,False,28.946022
1,patient0275,46.0,Intermediate_EF,70.181702,Higher_EF,False,24.181702
2,patient0182,50.0,Higher_EF,25.956150,Lower_EF,False,24.043850
3,patient0339,51.0,Higher_EF,28.283192,Lower_EF,False,22.716808
4,patient0194,67.0,Higher_EF,44.409175,Intermediate_EF,False,22.590825
5,patient0187,38.0,Lower_EF,55.049886,Intermediate_EF,False,17.049886
6,patient0037,49.0,Intermediate_EF,65.780810,Higher_EF,False,16.780810
7,patient0084,38.0,Lower_EF,53.897742,Intermediate_EF,False,15.897742
8,patient0322,43.0,Intermediate_EF,57.209817,Higher_EF,False,14.209817
9,patient0283,47.0,Intermediate_EF,57.997516,Higher_EF,False,10.997516



✓ Validation analysis saved.
✓ Test partition has still not been used.


In [ ]:
# ================================================================
# PHASE 3C — CELL 22
# FREEZE SELECTED CLASSIFIER
# ================================================================

import joblib


SELECTED_MODEL_PATH = (
    PHASE3C_MODEL_DIR /
    "phase3C_selected_logistic_regression.joblib"
)


# ---------------------------------------------------------------
# SAVE COMPLETE PIPELINE
# ---------------------------------------------------------------

joblib.dump(
    selected_phase3C_model,
    SELECTED_MODEL_PATH
)


# ---------------------------------------------------------------
# RELOAD IMMEDIATELY
# ---------------------------------------------------------------

reloaded_phase3C_model = (
    joblib.load(
        SELECTED_MODEL_PATH
    )
)


# ---------------------------------------------------------------
# VERIFY IDENTICAL VALIDATION PREDICTIONS
# ---------------------------------------------------------------

X_validation_selected = (
    validation_df[
        SELECTED_FEATURES
    ]
    .to_numpy(
        dtype=np.float64
    )
)


reloaded_predictions = (
    reloaded_phase3C_model.predict(
        X_validation_selected
    )
)


assert np.array_equal(
    reloaded_predictions,
    selected_validation_pred
), (
    "Reloaded model predictions differ "
    "from original selected model."
)


# ---------------------------------------------------------------
# MODEL HASH
# ---------------------------------------------------------------

SELECTED_MODEL_SHA256 = (
    calculate_sha256(
        SELECTED_MODEL_PATH
    )
)


print("=" * 80)
print("PHASE 3C — SELECTED MODEL FREEZE")
print("=" * 80)


print(
    "\nSelected feature set:",
    SELECTED_FEATURE_SET
)

print(
    "Number of features:",
    len(SELECTED_FEATURES)
)


print(
    "\nModel saved:"
)

print(
    SELECTED_MODEL_PATH
)


print(
    "\nSHA-256:"
)

print(
    SELECTED_MODEL_SHA256
)


print(
    "\n✓ Complete scaler + classifier pipeline saved."
)

print(
    "✓ Reloaded predictions exactly match."
)

print(
    "✓ Selected classifier artifact frozen."
)

PHASE 3C — SELECTED MODEL FREEZE

Selected feature set: F3_full_quantitative
Number of features: 8

Model saved:
/content/drive/MyDrive/CAMUS_FINAL/phase3_classification/phase3C_classifier_development/models/phase3C_selected_logistic_regression.joblib

SHA-256:
2ab03a57d0655aa557d0f3ce30c7438c84baa8e068575ba0ebc2ead3344194e6

✓ Complete scaler + classifier pipeline saved.
✓ Reloaded predictions exactly match.
✓ Selected classifier artifact frozen.


In [ ]:
# ================================================================
# PHASE 3C — CELL 23
# CREATE FROZEN DEVELOPMENT RECORD
# ================================================================

PHASE3C_RECORD_PATH = (
    PHASE3C_CONFIG_DIR /
    "phase3C_development_record.json"
)


# ---------------------------------------------------------------
# SELECTED VALIDATION METRICS
# ---------------------------------------------------------------

selected_accuracy = float(
    accuracy_score(
        y_validation,
        selected_validation_pred
    )
)

selected_balanced_accuracy = float(
    balanced_accuracy_score(
        y_validation,
        selected_validation_pred
    )
)

selected_macro_f1 = float(
    f1_score(
        y_validation,
        selected_validation_pred,
        average="macro",
        zero_division=0
    )
)

selected_weighted_f1 = float(
    f1_score(
        y_validation,
        selected_validation_pred,
        average="weighted",
        zero_division=0
    )
)

selected_kappa = float(
    cohen_kappa_score(
        y_validation,
        selected_validation_pred
    )
)


# ---------------------------------------------------------------
# ALL CANDIDATE RESULTS
# ---------------------------------------------------------------

candidate_results = []


for _, row in phase3C_results_df.iterrows():

    candidate_results.append({

        "feature_set":
            str(row["feature_set"]),

        "n_features":
            int(row["n_features"]),

        "accuracy":
            float(row["accuracy"]),

        "balanced_accuracy":
            float(row["balanced_accuracy"]),

        "macro_f1":
            float(row["macro_f1"]),

        "weighted_f1":
            float(row["weighted_f1"]),

        "cohen_kappa":
            float(row["cohen_kappa"])
    })


# ---------------------------------------------------------------
# RECORD
# ---------------------------------------------------------------

phase3C_record = {

    "phase":
        "Phase 3C — Learned Classifier Development",

    "status":
        "DEVELOPMENT_COMPLETE_AND_FROZEN",

    "development_data": {

        "train_patients":
            400,

        "validation_patients":
            50,

        "test_patients_used":
            0
    },

    "classification_target": {

        "type":
            "Frozen Phase 3A EF-based functional bands",

        "Lower_EF":
            "reference EF <= 40%",

        "Intermediate_EF":
            "40% < reference EF < 50%",

        "Higher_EF":
            "reference EF >= 50%"
    },

    "input_source":
        (
            "Frozen Phase 1 segmentation model followed by "
            "the frozen Phase 2 PCA area-length quantitative pipeline."
        ),

    "candidate_feature_sets":
        FEATURE_SETS,

    "classifier": {

        "pipeline":
            "StandardScaler + LogisticRegression",

        "solver":
            "lbfgs",

        "max_iter":
            5000,

        "random_state":
            int(SEED),

        "class_weight":
            None,

        "resampling":
            None
    },

    "selection": {

        "primary_metric":
            "validation macro F1",

        "tie_break":
            "fewer features for exact macro-F1 tie",

        "selected_feature_set":
            SELECTED_FEATURE_SET,

        "selected_features":
            SELECTED_FEATURES,

        "selected_n_features":
            len(SELECTED_FEATURES)
    },

    "candidate_validation_results":
        candidate_results,

    "selected_validation_results": {

        "accuracy":
            selected_accuracy,

        "balanced_accuracy":
            selected_balanced_accuracy,

        "macro_f1":
            selected_macro_f1,

        "weighted_f1":
            selected_weighted_f1,

        "cohen_kappa":
            selected_kappa,

        "correct":
            int(
                (
                    selected_validation_pred
                    ==
                    y_validation
                ).sum()
            ),

        "incorrect":
            int(
                (
                    selected_validation_pred
                    !=
                    y_validation
                ).sum()
            )
    },

    "selected_validation_confusion_matrix": {

        "class_order":
            CLASS_ORDER,

        "matrix":
            selected_confusion_matrix.tolist()
    },

    "selected_model_artifact": {

        "path":
            str(
                SELECTED_MODEL_PATH
            ),

        "sha256":
            SELECTED_MODEL_SHA256
    },

    "development_constraints": {

        "segmentation_model_modified":
            False,

        "quantitative_pipeline_modified":
            False,

        "classification_thresholds_modified":
            False,

        "test_data_used_for_selection":
            False,

        "post_validation_hyperparameter_search":
            False
    },

    "interpretation":
        (
            "Among the three predefined quantitative feature "
            "representations, F3_full_quantitative achieved the "
            "highest validation macro F1 and was therefore selected. "
            "The selected classifier is frozen before final test "
            "evaluation. Validation performance is moderate and "
            "does not establish clinical diagnostic performance."
        ),

    "phase3C_frozen":
        True
}


with open(
    PHASE3C_RECORD_PATH,
    "w"
) as file:

    json.dump(
        phase3C_record,
        file,
        indent=2
    )


print("=" * 80)
print("PHASE 3C DEVELOPMENT RECORD")
print("=" * 80)


print(
    "\nStatus:",
    phase3C_record["status"]
)


print(
    "Selected:",
    SELECTED_FEATURE_SET
)


print(
    "Validation accuracy:",
    f"{selected_accuracy:.4f}"
)

print(
    "Validation macro F1:",
    f"{selected_macro_f1:.4f}"
)


print(
    "\nTest patients used:",
    phase3C_record[
        "development_data"
    ][
        "test_patients_used"
    ]
)


print(
    "\nRecord saved:"
)

print(
    PHASE3C_RECORD_PATH
)


print(
    "\n✓ Phase 3C development decision recorded."
)

PHASE 3C DEVELOPMENT RECORD

Status: DEVELOPMENT_COMPLETE_AND_FROZEN
Selected: F3_full_quantitative
Validation accuracy: 0.4800
Validation macro F1: 0.4969

Test patients used: 0

Record saved:
/content/drive/MyDrive/CAMUS_FINAL/phase3_classification/phase3C_classifier_development/configs/phase3C_development_record.json

✓ Phase 3C development decision recorded.


In [ ]:
# ================================================================
# PHASE 3C — CELL 24
# FINAL FREEZE VERIFICATION
# ================================================================

print("=" * 80)
print("PHASE 3C — FINAL FREEZE VERIFICATION")
print("=" * 80)


# ---------------------------------------------------------------
# ARTIFACT EXISTENCE
# ---------------------------------------------------------------

required_phase3C_artifacts = {

    "Predefined protocol":
        PHASE3C_PROTOCOL_PATH,

    "Development feature table":
        FINAL_FEATURE_TABLE_PATH,

    "Validation comparison":
        RESULTS_PATH,

    "Per-class validation results":
        PER_CLASS_PATH,

    "Validation misclassifications":
        MISCLASSIFIED_PATH,

    "Selected classifier":
        SELECTED_MODEL_PATH,

    "Development record":
        PHASE3C_RECORD_PATH
}


all_artifacts_found = True


print("\nArtifacts:")


for name, path in (
    required_phase3C_artifacts.items()
):

    exists = path.exists()

    print(
        f"{'✓' if exists else '✗'} "
        f"{name}"
    )

    if not exists:
        print(
            "   ",
            path
        )

        all_artifacts_found = False


assert all_artifacts_found, (
    "Missing Phase 3C artifacts."
)


# ---------------------------------------------------------------
# RELOAD DEVELOPMENT RECORD
# ---------------------------------------------------------------

with open(
    PHASE3C_RECORD_PATH,
    "r"
) as file:

    frozen_record = (
        json.load(file)
    )


assert (
    frozen_record["status"]
    ==
    "DEVELOPMENT_COMPLETE_AND_FROZEN"
)


assert (
    frozen_record[
        "phase3C_frozen"
    ]
    is True
)


assert (
    frozen_record[
        "development_data"
    ][
        "test_patients_used"
    ]
    ==
    0
)


assert (
    frozen_record[
        "selection"
    ][
        "selected_feature_set"
    ]
    ==
    "F3_full_quantitative"
)


# ---------------------------------------------------------------
# VERIFY MODEL HASH AGAIN
# ---------------------------------------------------------------

current_model_hash = (
    calculate_sha256(
        SELECTED_MODEL_PATH
    )
)


assert (
    current_model_hash
    ==
    frozen_record[
        "selected_model_artifact"
    ][
        "sha256"
    ]
)


# ---------------------------------------------------------------
# VERIFY DEVELOPMENT TABLE HAS NO TEST PATIENTS
# ---------------------------------------------------------------

saved_development_df = (
    pd.read_csv(
        FINAL_FEATURE_TABLE_PATH
    )
)


assert len(
    saved_development_df
) == 450


assert not (
    saved_development_df[
        "split"
    ]
    ==
    "test"
).any()


# ---------------------------------------------------------------
# FINAL REPORT
# ---------------------------------------------------------------

print(
    "\n✓ All Phase 3C artifacts found."
)

print(
    "✓ Development record is frozen."
)

print(
    "✓ Selected model hash verified."
)

print(
    "✓ Selected feature set: "
    "F3_full_quantitative."
)

print(
    "✓ Development patients: 450."
)

print(
    "✓ Test patients used: 0."
)


print("\n" + "=" * 80)

print(
    "PHASE 3C STATUS: COMPLETE AND FROZEN"
)

print("=" * 80)


print(
    "\nNext phase:"
)

print(
    "Phase 3D — Final Locked Evaluation"
)

PHASE 3C — FINAL FREEZE VERIFICATION

Artifacts:
✓ Predefined protocol
✓ Development feature table
✓ Validation comparison
✓ Per-class validation results
✓ Validation misclassifications
✓ Selected classifier
✓ Development record

✓ All Phase 3C artifacts found.
✓ Development record is frozen.
✓ Selected model hash verified.
✓ Selected feature set: F3_full_quantitative.
✓ Development patients: 450.
✓ Test patients used: 0.

PHASE 3C STATUS: COMPLETE AND FROZEN

Next phase:
Phase 3D — Final Locked Evaluation
